# openai/privacy-filter laden und Layer inspizieren

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

c:\Users\Dexter\Desktop\pii_detection\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "openai/privacy-filter"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print(f"Device: {device}")
if device == "cuda":
    print(f"CUDA: {torch.cuda.get_device_name(0)}")

Device: cuda
CUDA: NVIDIA GeForce RTX 4070 Ti


In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    torch_dtype=dtype,
).to(device)

print(type(model))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 140/140 [00:00<00:00, 619.47it/s]


<class 'transformers.models.openai_privacy_filter.modeling_openai_privacy_filter.OpenAIPrivacyFilterForTokenClassification'>


In [4]:
# Gesamtes Modell anzeigen
print(model)

OpenAIPrivacyFilterForTokenClassification(
  (model): OpenAIPrivacyFilterModel(
    (embed_tokens): Embedding(200064, 640, padding_idx=199999)
    (layers): ModuleList(
      (0-7): 8 x OpenAIPrivacyFilterEncoderLayer(
        (self_attn): OpenAIPrivacyFilterAttention(
          (q_proj): Linear(in_features=640, out_features=896, bias=True)
          (k_proj): Linear(in_features=640, out_features=128, bias=True)
          (v_proj): Linear(in_features=640, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=640, bias=True)
        )
        (mlp): OpenAIPrivacyFilterMLP(
          (router): OpenAIPrivacyFilterTopKRouter()
          (experts): OpenAIPrivacyFilterExperts()
        )
        (input_layernorm): OpenAIPrivacyFilterRMSNorm((640,), eps=1e-05)
        (post_attention_layernorm): OpenAIPrivacyFilterRMSNorm((640,), eps=1e-05)
      )
    )
    (norm): OpenAIPrivacyFilterRMSNorm((640,), eps=1e-05)
    (rotary_emb): OpenAIPrivacyFilterRotaryEmbeddi

In [5]:
# Top-Level-Module
for name, module in model.named_children():
    print(f"{name}: {module.__class__.__name__}")

model: OpenAIPrivacyFilterModel
dropout: Dropout
score: Linear


In [6]:
# Detaillierte Layer-Namen (ersten 200)
for i, (name, module) in enumerate(model.named_modules()):
    if i >= 200:
        print("... (abgeschnitten)")
        break
    print(f"{i:03d}: {name} -> {module.__class__.__name__}")

000:  -> OpenAIPrivacyFilterForTokenClassification
001: model -> OpenAIPrivacyFilterModel
002: model.embed_tokens -> Embedding
003: model.layers -> ModuleList
004: model.layers.0 -> OpenAIPrivacyFilterEncoderLayer
005: model.layers.0.self_attn -> OpenAIPrivacyFilterAttention
006: model.layers.0.self_attn.q_proj -> Linear
007: model.layers.0.self_attn.k_proj -> Linear
008: model.layers.0.self_attn.v_proj -> Linear
009: model.layers.0.self_attn.o_proj -> Linear
010: model.layers.0.mlp -> OpenAIPrivacyFilterMLP
011: model.layers.0.mlp.router -> OpenAIPrivacyFilterTopKRouter
012: model.layers.0.mlp.experts -> OpenAIPrivacyFilterExperts
013: model.layers.0.input_layernorm -> OpenAIPrivacyFilterRMSNorm
014: model.layers.0.post_attention_layernorm -> OpenAIPrivacyFilterRMSNorm
015: model.layers.1 -> OpenAIPrivacyFilterEncoderLayer
016: model.layers.1.self_attn -> OpenAIPrivacyFilterAttention
017: model.layers.1.self_attn.q_proj -> Linear
018: model.layers.1.self_attn.k_proj -> Linear
019: mod

In [7]:
# Modellstatistiken: Gesamtparameter, trainierbare Parameter, dtypes, Speicherverbrauch
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

dtype_counts = {}
bytes_total = 0
for p in model.parameters():
    dt = str(p.dtype)
    dtype_counts[dt] = dtype_counts.get(dt, 0) + p.numel()
    bytes_total += p.numel() * p.element_size()

actual_dtype = max(dtype_counts, key=dtype_counts.get) if dtype_counts else "unknown"
memory_mib = bytes_total / (1024 ** 2)
memory_gib = bytes_total / (1024 ** 3)

print(f"Gesamtparameter:      {total_params:,}")
print(f"Trainierbar:          {trainable_params:,}")
print(f"Tatsaechlicher dtype: {actual_dtype}")
print(f"Speicher (Parameter): {memory_mib:,.2f} MiB ({memory_gib:,.2f} GiB)")

print("\nDtype-Verteilung:")
for dt, n in sorted(dtype_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"- {dt}: {n:,} Parameter")

Gesamtparameter:      1,399,486,865
Trainierbar:          1,399,486,865
Tatsaechlicher dtype: torch.float16
Speicher (Parameter): 2,669.31 MiB (2.61 GiB)

Dtype-Verteilung:
- torch.float16: 1,399,486,753 Parameter
- torch.float32: 112 Parameter
